# Sprint 11 · Webinar 35 · BI (Power BI + Tableau) · Student Version
## Modelado de datos, medidas avanzadas y dashboards interactivos

**Programa:** Data Analytics  
**Sprint:** 11  
**Webinar:** 35  
**Tipo de clase:** Teórico-práctica  
**Duración estimada:** 100 minutos

> En este cuaderno vas a completar apuntes, responder preguntas guía, registrar hallazgos y documentar los ejercicios realizados en **Power BI** y **Tableau**.


## Fecha

Completa la información de la sesión:

- **Fecha:** `____ / ____ / 2026`
- **Instructor/a:** `____________________________`
- **Grupo / Cohorte:** `____________________________`
- **Modalidad:** `Presencial / Virtual`


## Objetivos de la sesión

Al finalizar esta sesión, deberías poder:

1. Explicar con tus palabras qué es un **modelo en estrella** y por qué mejora el análisis.
2. Conectar varias fuentes de datos y definir relaciones correctamente en **Power BI** y **Tableau**.
3. Diferenciar **columnas calculadas**, **medidas** y **tablas calculadas** en DAX.
4. Reconocer el papel de **CALCULATE** en el contexto de filtro.
5. Construir análisis de **tiempo** y **cohortes**.
6. Diseñar dashboards pensando en distintos públicos de negocio.

### Mis objetivos personales para hoy
- `____________________________________________________________`
- `____________________________________________________________`


## Agenda sugerida (100 minutos)

| Tiempo | Bloque | Herramienta | Contenido | Mis notas / dudas |
|---:|---|:---:|---|---|
| 0–10 | Setup | 🐍 Python | Generación de los 5 datasets CSV | |
| 10–25 | Ejercicio 1 | Power BI + Tableau | Modelo de datos y relaciones | |
| 25–37 | Ejercicio 2 | Power BI | Columnas, medidas y tablas DAX | |
| 37–49 | Ejercicio 3 | Power BI | CALCULATE y contexto de filtro | |
| 49–61 | Ejercicio 4 | Power BI | Inteligencia de tiempo | |
| 61–74 | Ejercicio 5 | Power BI | Cohortes | |
| 74–84 | Ejercicio 6 | Tableau | LOD y cálculos comparables a DAX | |
| 84–94 | Ejercicio 7 | Tableau | Tiempo y cohortes | |
| 94–100 | Cierre | Power BI + Tableau | Dashboard final y takeaways | |


---

# Setup · Generación de los datasets (10 min) 🐍

**Meta:** producir los archivos CSV que usarás durante toda la sesión.

## Antes de ejecutar
Responde brevemente:

1. **¿Por qué conviene generar o revisar los datos antes de abrir la herramienta BI?**  
   `____________________________________________________________`

2. **¿Qué ventajas tiene trabajar con varias tablas en vez de una sola tabla gigante?**  
   `____________________________________________________________`

3. **¿Qué tablas esperas encontrar en un modelo tipo estrella?**  
   `____________________________________________________________`


### Imports y configuración

Ejecuta la siguiente celda primero.

### Checklist
- [ ] La celda corrió sin errores.
- [ ] Entiendo en qué carpeta se guardarán los archivos.
- [ ] Si hubo un error, lo anoté aquí:

`____________________________________________________________`


In [ ]:
# ============================================================
# Imports y configuración — ejecuta esta celda primero
# ============================================================

import numpy as np
import pandas as pd
import os

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

OUTPUT_DIR = 'dataset_sprint11_w35'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('✅ Librerías cargadas.')
print('📁 Carpeta de salida:', OUTPUT_DIR)


### Generador de datos

Ejecuta esta celda para crear los datasets de la sesión.

### Antes de ejecutar
Completa esta predicción:

- **Número de tablas que se crearán:** `________`
- **Tabla que probablemente será la tabla de hechos:** `________________`
- **Dos tablas que probablemente actuarán como dimensiones:** `________________` y `________________`


In [ ]:
# ============================================================
# GENERADOR DE DATASETS — Webinar 35 Sprint 11
# Ejecuta este bloque completo para obtener los 5 CSV
# ============================================================

rng = np.random.default_rng(RANDOM_SEED)

# ── 1. DIMENSIÓN: CLIENTES ───────────────────────────────────
N_CLIENTES = 500
ciudades  = ['Bogotá','Medellín','Cali','Barranquilla','Cartagena','Bucaramanga','Pereira']
segmentos = ['Consumidor','Corporativo','PYME']
generos   = ['M','F','No especificado']

clientes = pd.DataFrame({
    'cliente_id'    : [f'CLI{str(i).zfill(4)}' for i in range(1, N_CLIENTES+1)],
    'nombre'        : [f'Cliente_{i}' for i in range(1, N_CLIENTES+1)],
    'ciudad'        : rng.choice(ciudades, N_CLIENTES, p=[.30,.22,.18,.12,.08,.05,.05]),
    'segmento'      : rng.choice(segmentos, N_CLIENTES, p=[.55,.25,.20]),
    'genero'        : rng.choice(generos, N_CLIENTES, p=[.45,.45,.10]),
    'fecha_registro': pd.to_datetime(
        rng.integers(
            pd.Timestamp('2021-01-01').value,
            pd.Timestamp('2023-06-30').value, N_CLIENTES)
    ).normalize(),
    'edad': rng.integers(18, 65, N_CLIENTES),
})

# ── 2. DIMENSIÓN: PRODUCTOS ──────────────────────────────────
N_PRODUCTOS = 80
categorias = {
    'Electrónica': ['Smartphones','Laptops','Accesorios','Audio'],
    'Hogar'      : ['Muebles','Decoración','Cocina'],
    'Ropa'       : ['Ropa Hombre','Ropa Mujer','Calzado'],
    'Deportes'   : ['Fitness','Outdoor','Ciclismo'],
}
cat_list, sub_list = [], []
for cat, subs in categorias.items():
    n = N_PRODUCTOS // len(categorias)
    for sub in subs:
        k = n // len(subs)
        cat_list.extend([cat]*k); sub_list.extend([sub]*k)
while len(cat_list) < N_PRODUCTOS:
    cat_list.append('Electrónica'); sub_list.append('Accesorios')

precio_lista = np.round(rng.uniform(15, 600, N_PRODUCTOS), 2)
margen_ratio = rng.uniform(0.35, 0.60, N_PRODUCTOS)

productos = pd.DataFrame({
    'producto_id'    : [f'PRD{str(i).zfill(3)}' for i in range(1, N_PRODUCTOS+1)],
    'nombre_producto': [f'{sub_list[i-1]}_{i}' for i in range(1, N_PRODUCTOS+1)],
    'categoria'      : cat_list[:N_PRODUCTOS],
    'subcategoria'   : sub_list[:N_PRODUCTOS],
    'precio_lista'   : precio_lista,
    'costo_unitario' : np.round(precio_lista * (1 - margen_ratio), 2),
})

# ── 3. DIMENSIÓN: CALENDARIO ─────────────────────────────────
fechas = pd.date_range('2022-01-01', '2024-03-31', freq='D')
calendario = pd.DataFrame({
    'fecha'        : fechas,
    'año'          : fechas.year,
    'trimestre'    : fechas.quarter,
    'mes_num'      : fechas.month,
    'mes_nombre'   : fechas.month_name(),
    'semana'       : fechas.isocalendar().week.astype(int),
    'dia_semana'   : fechas.day_name(),
    'es_fin_semana': (fechas.dayofweek >= 5).astype(int),
    'año_mes'      : fechas.to_period('M').astype(str),
    'año_trimestre': fechas.year.astype(str) + '-Q' + fechas.quarter.astype(str),
})

# ── 4. TABLA DE HECHOS: VENTAS ───────────────────────────────
N_VENTAS = 12_000
canales  = ['Online','Tienda Física','App Móvil','Marketplace']

# Fechas con estacionalidad: Q4 y mitad de año tienen más ventas
fecha_vals = pd.date_range('2022-01-01', '2024-03-31', freq='D')
pesos = np.where(fecha_vals.month.isin([11,12]), 2.5,
         np.where(fecha_vals.month.isin([6,7]), 1.4, 1.0))
pesos = pesos / pesos.sum()

fechas_venta = rng.choice(fecha_vals, size=N_VENTAS, p=pesos)
cli_idx  = rng.choice(clientes['cliente_id'], N_VENTAS)
prod_idx = rng.choice(productos['producto_id'], N_VENTAS,
                       p=rng.dirichlet(np.ones(N_PRODUCTOS)))
precios_ref = dict(zip(productos['producto_id'], productos['precio_lista']))
precio_real = np.array([precios_ref[p] for p in prod_idx])

ventas = pd.DataFrame({
    'order_id'       : [f'ORD{str(i).zfill(6)}' for i in range(1, N_VENTAS+1)],
    'cliente_id'     : cli_idx,
    'producto_id'    : prod_idx,
    'fecha'          : pd.to_datetime(fechas_venta),
    'cantidad'       : rng.integers(1, 6, N_VENTAS),
    'precio_unitario': np.round(precio_real, 2),
    'descuento_pct'  : np.round(
        rng.choice([0,.05,.10,.15,.20], N_VENTAS, p=[.50,.20,.15,.10,.05]), 2),
    'canal'          : rng.choice(canales, N_VENTAS, p=[.45,.25,.20,.10]),
})
ventas['ingreso_neto'] = np.round(
    ventas['cantidad'] * ventas['precio_unitario'] * (1 - ventas['descuento_pct']), 2
)

# ── 5. DEVOLUCIONES ──────────────────────────────────────────
idx_dev = rng.choice(N_VENTAS, size=int(N_VENTAS * 0.06), replace=False)
motivos = ['Defecto de fábrica','No era lo esperado','Llegó tarde',
           'Talla incorrecta','Duplicado']

devoluciones = pd.DataFrame({
    'devolucion_id'   : [f'DEV{str(i).zfill(5)}' for i in range(1, len(idx_dev)+1)],
    'order_id'        : ventas.iloc[idx_dev]['order_id'].values,
    'fecha_devolucion': (
        pd.to_datetime(ventas.iloc[idx_dev]['fecha'].values)
        + pd.to_timedelta(rng.integers(2, 30, len(idx_dev)), unit='D')
    ),
    'motivo'          : rng.choice(motivos, len(idx_dev), p=[.30,.28,.18,.14,.10]),
    'cantidad_devuelta': rng.integers(1, 4, len(idx_dev)),
})

# ── Guardar CSVs ─────────────────────────────────────────────
clientes.to_csv(f'{OUTPUT_DIR}/clientes.csv',        index=False)
productos.to_csv(f'{OUTPUT_DIR}/productos.csv',      index=False)
calendario.to_csv(f'{OUTPUT_DIR}/calendario.csv',    index=False)
ventas.to_csv(f'{OUTPUT_DIR}/ventas.csv',            index=False)
devoluciones.to_csv(f'{OUTPUT_DIR}/devoluciones.csv',index=False)

print('✅ Datasets generados exitosamente en:', OUTPUT_DIR)
print()
print(f'  📄 clientes.csv      → {len(clientes):>6,} filas | columnas: {list(clientes.columns)}')
print(f'  📄 productos.csv     → {len(productos):>6,} filas | columnas: {list(productos.columns)}')
print(f'  📄 calendario.csv    → {len(calendario):>6,} filas | columnas: {list(calendario.columns)}')
print(f'  📄 ventas.csv        → {len(ventas):>6,} filas | columnas: {list(ventas.columns)}')
print(f'  📄 devoluciones.csv  → {len(devoluciones):>6,} filas | columnas: {list(devoluciones.columns)}')
print()
print('→ Abre Power BI Desktop o Tableau Public y carga estos archivos para continuar.')


### Vista previa de los datasets

Ejecuta la siguiente celda y registra lo que observes.

### Observaciones rápidas
- **Archivo con transacciones o eventos:** `____________________________`
- **Archivo con información de clientes:** `____________________________`
- **Archivo con información de productos:** `____________________________`
- **Columna candidata para relacionar ventas con clientes:** `________________`
- **Columna candidata para relacionar ventas con productos:** `_______________`


In [ ]:
# Vista previa — últimas celdas Python de la sesión
print('=== ventas.csv — primeras 5 filas ===')
display(ventas.head())

print('\n=== clientes.csv — primeras 5 filas ===')
display(clientes.head())

print('\n=== productos.csv — primeras 5 filas ===')
display(productos.head())

print('\n=== calendario.csv — primeras 5 filas ===')
display(calendario.head())

print('\n=== devoluciones.csv — primeras 5 filas ===')
display(devoluciones.head())


---

# Ejercicio 1 · 11.1 Conexión de múltiples fuentes y modelado de relaciones (15 min)

**Meta:** importar las 5 tablas en Power BI y Tableau, configurar relaciones y documentar la lógica del modelo.

## Lo que debes lograr
- Importar todos los CSV.
- Identificar **hechos** y **dimensiones**.
- Definir claves correctas.
- Detectar posibles problemas de modelado.


## 11.1.1 · Fundamentos del modelado

### Completa con tus palabras

| Término | Mi definición |
|---|---|
| **Tabla de hechos** | |
| **Tabla de dimensión** | |
| **Clave primaria** | |
| **Clave foránea** | |
| **Granularidad** | |
| **Modelo en estrella** | |

### Reflexión breve
1. **¿Cuál crees que es la granularidad de la tabla de ventas?**  
   `____________________________________________________________`

2. **¿Por qué una tabla calendario suele ser útil?**  
   `____________________________________________________________`


## 11.1.2 · Relaciones y comportamiento del modelo en Power BI

### Paso a paso guiado
1. Abre **Power BI Desktop**.
2. Importa los 5 archivos CSV.
3. Ve a la vista **Modelo**.
4. Identifica la tabla central.
5. Crea las relaciones necesarias.
6. Revisa la cardinalidad y la dirección del filtro.

### Registro del ejercicio
| Elemento | Respuesta del estudiante |
|---|---|
| Tabla central del modelo | |
| Relación clientes ↔ ventas | |
| Relación productos ↔ ventas | |
| Relación calendario ↔ ventas | |
| ¿La cardinalidad fue 1:*? ¿Dónde? | |
| ¿Hubo alguna relación ambigua? | |

### Preguntas de comprensión
- **¿Qué pasa si conectas dos tablas dimensión entre sí sin necesidad?**  
  `____________________________________________________________`

- **¿Qué riesgo hay si la columna que debería ser clave tiene duplicados?**  
  `____________________________________________________________`


## 11.1.3 · Aplicando el modelado de datos en Tableau

### Compara ambas herramientas

| Aspecto | Power BI | Tableau | Mi observación |
|---|---|---|---|
| Vista de relaciones | | | |
| Forma de modelar | | | |
| Nivel lógico / físico | | | |
| Facilidad para detectar errores | | | |

### Después de practicar en Tableau
- **¿Qué fue más intuitivo en Tableau?**  
  `____________________________________________________________`
- **¿Qué fue más claro en Power BI?**  
  `____________________________________________________________`
- **¿En cuál herramienta entendiste mejor el modelo? ¿por qué?**  
  `____________________________________________________________`


In [ ]:

# Espacio del estudiante
# Usa esta celda para dejar notas de modelado, nombres de columnas clave
# o dificultades encontradas durante el ejercicio en Power BI/Tableau.

# Ejemplo:
# - ventas conecta con clientes por:
# - ventas conecta con productos por:
# - duda principal:


---

# Ejercicio 2 · 11.2.1 Creación de cálculos en DAX: columnas, medidas y tablas (12 min)

**Meta:** distinguir claramente cuándo usar cada tipo de objeto en DAX.

### Antes de empezar
Marca tu hipótesis inicial:
- [ ] Una columna calculada se evalúa fila por fila.
- [ ] Una medida responde al contexto del visual.
- [ ] Una tabla calculada sirve para crear nuevas tablas derivadas.


## Fundamentos teóricos

Completa la tabla durante la explicación:

| Tipo | ¿Dónde vive? | ¿Cuándo se calcula? | Caso de uso | Ejemplo propio |
|---|---|---|---|---|
| **Columna calculada** | | | | |
| **Medida** | | | | |
| **Tabla calculada** | | | | |

### Aplica la idea
Clasifica cada necesidad:

1. **Crear el margen por cada fila de venta** → `____________________`
2. **Calcular ventas totales del dashboard** → `____________________`
3. **Construir una tabla resumen aparte** → `____________________`

### Nota personal
`____________________________________________________________`


In [ ]:

# Espacio del estudiante
# Escribe aquí ejemplos de fórmulas DAX que hayas creado en Power BI.
# También puedes copiar una medida y explicar qué hace.

# Medida 1:
# Explicación:
#
# Columna calculada:
# Explicación:


---

# Ejercicio 3 · 11.2.2 Conociendo CALCULATE: el motor de DAX (12 min)

**Meta:** comprender cómo cambia el resultado de una medida cuando cambia el contexto de filtro.

### Predicción
Antes de ver los ejemplos, responde:

- **¿Qué crees que hace CALCULATE?**  
  `____________________________________________________________`

- **¿Qué entiendes por contexto de filtro?**  
  `____________________________________________________________`


## Fundamentos teóricos

### Estructura general
```dax
CALCULATE(<expresión>, <filtro1>, <filtro2>, ...)
```

### Completa esta tabla
| Elemento | ¿Qué hace? |
|---|---|
| `<expresión>` | |
| `<filtro>` | |
| Cambio de contexto | |

### Ejemplos trabajados en clase
Anota al menos 2:

1. `__________________________________________________________`
2. `__________________________________________________________`

### Después de practicar
- **¿Qué diferencia viste entre una medida simple y una medida con CALCULATE?**  
  `____________________________________________________________`
- **¿Qué función te resultó más útil junto con CALCULATE?**  
  `____________________________________________________________`


In [ ]:

# Espacio del estudiante
# Copia aquí una o dos medidas con CALCULATE y explica con comentarios
# qué filtros están entrando en juego.

# Ejemplo:
# Ventas Online =
# CALCULATE(...)
#
# Explicación:


---

# Ejercicio 4 · 11.2.3 Analizando datos en el tiempo con DAX (12 min)

**Meta:** construir métricas temporales y entender sus diferencias.

### Antes de practicar
Completa con tu intuición:

| Métrica | ¿Qué crees que significa? |
|---|---|
| **YTD** | |
| **MTD** | |
| **QoQ** | |
| **YoY** | |

### Requisito importante
- **¿Por qué este tipo de análisis suele necesitar una tabla calendario?**  
  `____________________________________________________________`


## Fundamentos teóricos

Registra aquí las fórmulas o ideas clave vistas en clase:

### Funciones / enfoques mencionados
- `____________________________________________________________`
- `____________________________________________________________`
- `____________________________________________________________`

### Comparación de métricas temporales
| Cálculo | ¿Qué compara? | ¿Para qué sirve? |
|---|---|---|
| YTD | | |
| Variación mensual | | |
| Variación trimestral | | |
| Variación anual | | |

### Interpretación
Escribe un ejemplo de insight temporal que obtuviste:
`____________________________________________________________`


In [ ]:

# Espacio del estudiante
# Anota aquí medidas de inteligencia temporal creadas en Power BI.

# Medida:
# Qué calcula:
# Resultado observado:


---

# Ejercicio 5 · 11.2.4 Analizando y visualizando cohortes en Power BI (13 min)

**Meta:** construir una matriz o visual que permita analizar retención por cohorte.

### Antes de empezar
Responde:

- **¿Qué es una cohorte?**  
  `____________________________________________________________`
- **¿Cuál podría ser el evento de inicio de una cohorte en este caso?**  
  `____________________________________________________________`
- **¿Qué diferencia hay entre adquisición y retención?**  
  `____________________________________________________________`


## Fundamentos teóricos

### Estructura del análisis de cohortes
Completa:

| Elemento | Mi respuesta |
|---|---|
| Fecha de inicio de cohorte | |
| Unidad de análisis | |
| Métrica de retención | |
| Periodo de seguimiento | |

### Durante la práctica
Anota los pasos que seguiste:
1. `__________________________________________________________`
2. `__________________________________________________________`
3. `__________________________________________________________`
4. `__________________________________________________________`

### Interpretación del resultado
- **¿Qué cohorte retuvo mejor?** `____________________________`
- **¿En qué periodo cae más la retención?** `___________________`
- **¿Qué hipótesis de negocio propones?**  
  `____________________________________________________________`


In [ ]:

# Espacio del estudiante
# Registra aquí observaciones del heatmap o la matriz de cohortes.

# Cohorte destacada:
# Caída más fuerte:
# Posible explicación:


---

# Ejercicio 6 · 11.2.5 Creando cálculos en Tableau: de CALCULATE a LOD (10 min)

**Meta:** entender cómo Tableau resuelve cálculos agregados con expresiones LOD.

### Antes de ver el ejemplo
- **¿Qué recuerdas sobre LOD (Level of Detail)?**  
  `____________________________________________________________`
- **¿Por qué Tableau necesita controlar el nivel de detalle?**  
  `____________________________________________________________`


## Fundamentos teóricos

### Tipos de expresiones LOD
| Tipo | ¿Qué hace? | Ejemplo o nota |
|---|---|---|
| **FIXED** | | |
| **INCLUDE** | | |
| **EXCLUDE** | | |

### Puente conceptual con Power BI
| Idea en Power BI / DAX | Aproximación en Tableau | Mi explicación |
|---|---|---|
| Cambiar contexto de filtro | | |
| Calcular a otro nivel de agregación | | |
| Mantener un valor estable sin depender del visual | | |

### Después de la práctica
- **¿Qué expresión LOD te pareció más clara?** `___________________`
- **¿Qué te pareció más difícil al traducir desde DAX?**  
  `____________________________________________________________`


In [ ]:

# Espacio del estudiante
# Escribe aquí cálculos de Tableau o notas sobre FIXED / INCLUDE / EXCLUDE.

# Cálculo:
# Qué hace:
# Cuándo lo usaría:


---

# Ejercicio 7 · 11.2.6 Analizando tiempo y cohortes en Tableau (10 min)

**Meta:** replicar el razonamiento analítico de tiempo y cohortes en Tableau.

### Registro comparativo
| Tema | ¿Cómo lo resolviste en Power BI? | ¿Cómo lo resolviste en Tableau? |
|---|---|---|
| Inteligencia de tiempo | | |
| Cohortes | | |
| Campo calculado | | |
| Visualización final | | |

### Reflexión
- **¿Qué análisis te resultó más directo en Tableau?**  
  `____________________________________________________________`
- **¿Qué análisis prefieres hacer en Power BI?**  
  `____________________________________________________________`


In [ ]:

# Espacio del estudiante
# Usa esta celda para documentar fórmulas, dimensiones calculadas
# o pasos seguidos en Tableau.


---

# Ejercicio 8 · 11.3 Diseño de paneles interactivos para diferentes públicos (15 min)

**Meta:** diseñar un dashboard que responda a una audiencia específica.

## Fundamentos teóricos

### ¿Quién es tu audiencia?
Elige una:
- [ ] Dirección / liderazgo
- [ ] Equipo comercial
- [ ] Marketing
- [ ] Operaciones
- [ ] Producto

### Diseño propuesto
| Elemento | Mi decisión |
|---|---|
| Audiencia objetivo | |
| Pregunta principal que debe responder el dashboard | |
| KPI más importante | |
| Visual 1 | |
| Visual 2 | |
| Visual 3 | |
| Filtros o segmentadores | |
| Acción interactiva esperada del usuario | |

### Principios de diseño
Marca los que aplicaste:
- [ ] Jerarquía visual
- [ ] Pocos KPIs clave
- [ ] Consistencia de colores
- [ ] Títulos claros
- [ ] Filtros útiles
- [ ] Evitar saturación de elementos

### Retroalimentación
- **¿Qué mejorarías de tu panel si tuvieras 20 minutos más?**  
  `____________________________________________________________`


In [ ]:

# Espacio del estudiante
# Boceto textual del dashboard final

# Título:
# Público:
# KPIs:
# Visuales:
# Filtros:
# Insight principal:


---

# Cierre y Takeaways (5 min)

## Completa las ideas clave de la sesión

1. Un **modelo en estrella** me ayuda a `________________________________________`
2. Una **medida** es diferente de una columna calculada porque `___________________`
3. **CALCULATE** sirve para `_______________________________________________`
4. La inteligencia de tiempo necesita `____________________________________`
5. Un análisis de **cohortes** me permite `_________________________________`
6. En **Tableau**, las expresiones **LOD** sirven para `_____________________`
7. Un buen dashboard debe adaptarse a `____________________________________`


## Cierre personal

### Autoevaluación
Marca cómo te sentiste hoy:
- [ ] Puedo explicar los conceptos principales sin ayuda.
- [ ] Entiendo la lógica general, pero debo practicar.
- [ ] Necesito repasar modelado de datos.
- [ ] Necesito repasar DAX.
- [ ] Necesito repasar Tableau.

### Lo que me llevo de la sesión
- **Lo más claro de hoy fue:** `___________________________________________`
- **Lo más retador fue:** `_______________________________________________`
- **Pregunta que todavía tengo:** `________________________________________`
- **Tema que quiero practicar después:** `________________________________`


## Notas finales del estudiante

Usa este espacio para apuntes libres, fórmulas, conceptos, dudas o pasos que quieras recordar:

`____________________________________________________________`

`____________________________________________________________`

`____________________________________________________________`

`____________________________________________________________`

### ¿Necesitas ayuda?
Recuerda los canales de ayuda que tenemos disponibles:
- `DATA-CO-LEARNING`: Revisa los horarios de atencion en la guia dele studiante. Recuerda que tenemos horarios de apoyo todos los días para tus dudas puntuales!
- `DA_CONSULTA`: Uuedes publicar tus preguntas sobre el contenido de la plataforma o el proyecto 24/7. Recuerda que en tu pregunta debes de agregar el tag @Dataconsulta.
- `SPRINT FOCUS`: Para los Sprints 1 al 5 tenemos sesiones extra donde abordamos temáticas de cada sprint a rpofundidad. Revisa la guia del estudiante para conocer los horarios!
- `SESIONES 1:1`: Recuerda que puedes agendar sesiones 1:1 con un tutor. Agendalas con antelación y resuelve todas tus dudas, desde temás del proyecto, curso, carrera hasta problemas técnicos.
- `CANAL DE DISCORD DE COMMUNITY`: Recuerda que tu cohorte tiene un canal especial donde puedes compartir cualquier item interesante que quieras mostrar a tus compañeros de curso.
